# Preparo

## Bibliotecas

In [49]:
import os
import sys

import pandas as pd
import plotly.express as px

In [50]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [51]:
from utils.add_country_name import add_country_name
from utils.adjust_cuisines import adjust_cuisines
from utils.clean_data import clean_data
from utils.convert_to_usd import convert_to_usd
from utils.create_color_name import create_color_name
from utils.create_price_tye import create_price_tye
from utils.create_unique_restaurant_name import create_unique_restaurant_name
from utils.rename_columns import rename_columns

## Baixando o Dataset

In [52]:
file_path = os.path.join("..", "database", "zomato.csv")

print(f"Tentando ler o arquivo em: {os.path.abspath(file_path)}")

try:
    df = pd.read_csv(file_path, 
                    encoding="utf-8", 
                    on_bad_lines="skip")
    print("Sucesso! O DataFrame foi carregado.")
except Exception as e:
    print(f"Erro: {e}")

Tentando ler o arquivo em: c:\Users\Admin\Documents\Comunidade DS\Analise de Dados Com Python\Empresa Fome Zero\database\zomato.csv
Sucesso! O DataFrame foi carregado.


## Limpando o Dataset

In [53]:
df = (df.pipe(rename_columns)
                .pipe(add_country_name)
                .pipe(clean_data)
                .pipe(convert_to_usd)
                .pipe(create_price_tye)
                .pipe(create_color_name)
                .pipe(adjust_cuisines)
                .pipe(create_unique_restaurant_name))

# Cidade

## Top 10 Cidades com mais Restaurantes na Base de Dados

In [54]:
cidade_quantidade_restaurantes = df.groupby("city").size().reset_index(name="quantidade").sort_values(by="quantidade", ascending=False)

top_10_cidades_com_mais_restaurantes = cidade_quantidade_restaurantes.head(10)

fig = px.bar(
    top_10_cidades_com_mais_restaurantes, 
    x="city", 
    y="quantidade", 
    title="Top 10 Cidades com Mais Restaurantes",
    labels={"city": "Cidade", "quantidade": "Quantidade de Restaurantes."},
    text_auto=True, # Adiciona o número de restaurantes em cima de cada barra
    color="city", # Opcional: cria um degradê de cores baseado na quantidade
    color_discrete_sequence=px.colors.qualitative.Safe # Cores para pessoas daltonicas
)

# Melhorando o layout para facilitar a leitura
fig.update_layout(
    xaxis_tickangle=-45, # Inclina os nomes das cidades para não sobreporem
    title_x=0.5, # Centraliza o título
    xaxis_categoryorder='total descending',
)

# Exibindo o gráfico
fig.show()

## Top 7 Cidades com Restaurantes com média de avaliação acima de 4

In [55]:
filtro = df["aggregate_rating"] >= 4

cidade_quantidade_restaurantes_nota_4 = df.loc[filtro, :].groupby(["country", "city"]).size().reset_index(name="quantidade").sort_values(by="quantidade", ascending=False)

top_7_cidades_restaurantes_media_acima_4 = cidade_quantidade_restaurantes_nota_4.head(7)

# 2. Criando o gráfico de barras
fig = px.bar(
    top_7_cidades_restaurantes_media_acima_4, 
    x="city", 
    y="quantidade", 
    title="Top 7 Cidades com Restaurantes (Avaliação > 4)",
    labels={"city": "Cidade", "quantidade": "Quantidade de Restaurantes", "country": "País"},
    text_auto=True,
    color="country",
    color_discrete_sequence=px.colors.qualitative.Safe # Mantendo a paleta acessível!
)

# 3. Melhorando o visual (inclinando texto e removendo legenda)
fig.update_layout(
    xaxis_tickangle=-45, 
    title_x=0.5,
    xaxis_categoryorder='total descending',
)

# 4. Exibindo o gráfico
fig.show()


## Top 7 Cidades com Restaurantes com Média de Avaliação Abaixo de 2.5

In [56]:
filtro = df["aggregate_rating"] <= 2.5

cidade_quantidade_restaurantes_nota_2_5 = df.loc[filtro, :].groupby(["country", "city"]).size().reset_index(name="quantidade").sort_values(by="quantidade", ascending=False)

top_7_cidades_restaurantes_media_abaixo_2_5 = cidade_quantidade_restaurantes_nota_2_5.head(7)

# 2. Criando o gráfico de barras
fig = px.bar(
    top_7_cidades_restaurantes_media_abaixo_2_5, 
    x="city", 
    y="quantidade", 
    title="Top 7 Cidades com Restaurantes (Avaliação < 2.5)",
    labels={"city": "Cidade", "quantidade": "Quantidade de Restaurantes", "country": "País"},
    text_auto=True,
    color="country",
    color_discrete_sequence=px.colors.qualitative.Safe # Mantendo a paleta acessível
)

# 3. Ajustando o visual
fig.update_layout(
    xaxis_tickangle=-45, 
    title_x=0.5, 
    xaxis_categoryorder='total descending',
)

# 4. Exibindo o gráfico
fig.show()

## Top 10 Cidades mais Restaurantes com Tipos Culinários Distintos

In [57]:
# 1. Agrupando por país e cidade, e contando os tipos culinários distintos (valores únicos)
cidades_tipos_culinarios = (
    df.groupby(["country", "city"])["cuisines"]
    .nunique() # Conta quantos tipos culinários diferentes existem
    .reset_index(name="quantidade_tipos_culinarios")
    .sort_values(by="quantidade_tipos_culinarios", ascending=False)
)

# 2. Pegando o Top 10
top_10_cidades_culinaria_distintos = cidades_tipos_culinarios.head(10)

# 3. Criando o gráfico
fig = px.bar(
    top_10_cidades_culinaria_distintos, 
    x="city", 
    y="quantidade_tipos_culinarios", 
    title="Top 10 Cidades com Mais Tipos Culinários Distintos",
    labels={
        "city": "Cidade", 
        "quantidade_tipos_culinarios": "Tipos Culinários Únicos",
        "country": "País"
    },
    text_auto=True,
    color="country", # Cores separadas por país
    color_discrete_sequence=px.colors.qualitative.Safe # Paleta segura para daltônicos
)

# 4. Ajustando o visual
fig.update_layout(
    xaxis_tickangle=-45, 
    title_x=0.5,
    xaxis_categoryorder='total descending'
)

# 5. Exibindo o gráfico
fig.show()